In [1]:
# Packages to Install for Scraping
!pip -q install requests beautifulsoup4 
import requests, json
from bs4 import BeautifulSoup
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlip
from pathlib import Path

In [2]:
# Fetch the HTML content
url = "https://www.boston.gov/public-notices/16600326"
response = requests.get(url)
#print(response.text)
soup = BeautifulSoup(response.text, 'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="og: https://ogp.me/ns#">
 <head>
  <meta charset="utf-8"/>
  <link href="https://www.boston.gov/public-notices/16600326" rel="canonical"/>
  <link href="https://www.boston.gov/node/16600326" rel="shortlink"/>
  <meta content="Drupal 10 (http://drupal.org)" name="generator"/>
  <link href="/themes/custom/bos_theme/images/apple-touch-icon.png" rel="apple-touch-icon" sizes="180x180"/>
  <link href="/themes/custom/bos_theme/images/apple-touch-icon-precomposed.png" rel="apple-touch-icon-precomposed" sizes="180x180"/>
  <meta content="Boston.gov" property="og:site_name"/>
  <meta content="https://www.boston.gov/public-notices/16600326" property="og:url"/>
  <meta content="City Council Committee on Civil Rights, Racial Equity, and Immigrant Advancement" property="og:title"/>
  <meta content="https://patterns.boston.gov/images/global/icons/seal_dark_1000x1000.png" property="og:image"/>
  <meta content="2026-07-15T15:31:31-0400" property="og:upd

In [10]:
title=soup.title.string
detail_url = url
notice_id = 16600326

# Finding the Posted Date
posted_label=soup.find("div",class_="dl-t", string=lambda t:t and "Posted" in t)
posted_raw=posted_label.find_next_sibling("div",class_="dl-d").get_text(strip=True)
posted_at = datetime.strptime(posted_raw, "%m/%d/%Y - %I:%M%p").replace(tzinfo=ZoneInfo("America/New_York")).isoformat()

# Discussion Topics Text
discussion_label = soup.find("h2",class_="header-border-bottom", string=lambda t:t and "Discussion Topics" in t)
discussion_text = discussion_label.find_next_sibling("div",class_="body").get_text(strip=False)


# Event details
event_date_container = soup.find("div", class_="date-title")
event_datetime = event_date_container.find("time")["datetime"]
address_container = soup.find("div",class_="detail-item__body--secondary sb-d")
address_line_1 = address_container.find("span",class_="address-line1").get_text(strip=False)
address_line_2 = address_container.find("span",class_="address-line2").get_text(strip=False)

#Look for public comment
public_testimony = False
testimony = soup.find("div",class_="n-li-a", string=lambda t:t and "The public can offer testimony" in t)
if testimony:
    public_testimony = True

# PDFS
files = []
resources_label = soup.find("div", class_="sb-t", string=lambda t: t and "Resources" in t)
resources_container = resources_label.find_parent("div", class_="detail-item__content")
pdf_links = resources_container.select("div.link-wrapper.download-link a")

files = [{"file_label": a.get_text(strip=True), "file_url": a["href"]} for a in pdf_links]


#Print for double checking
print(title)
print(posted_at)
print(event_datetime)
print(address_line_1)
print(address_line_2)
print(public_testimony)
print(files)

record = {
    "notice_id": notice_id,
    "title": title,
    "detail_url": detail_url,
    "posted_at": posted_at,
    "event_datetime": event_datetime,
    "address_1":address_line_1,
    "address_2":address_line_2,
    "page_text": discussion_items,
    "files": files,
    "status": "ok",
    "checked_at": datetime.now(timezone.utc).isoformat(),
}


City Council Committee on Civil Rights, Racial Equity, and Immigrant Advancement | Boston.gov
2026-07-15T15:17:00-04:00
2026-07-21T18:00:00Z
Iannella Chamber, 5th Floor
Boston City Hall
True
[{'file_label': 'Docket #1237', 'file_url': 'https://www.boston.gov/sites/default/files/file/2026/07/Human%20Rights%20Commission%20Hearing%20Order%20%288%29.pdf'}, {'file_label': 'Official Filed Posting', 'file_url': 'https://www.boston.gov/sites/default/files/file/2026/07/CityCouncil.7.15.2026.pdf'}]
